## 1. Setup check (read-only)
Confirms the packages and settings this notebook needs.

In [ ]:
import importlib.util
import os

def _has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except ModuleNotFoundError:
        return False

packages = {"pdfplumber": "pdfplumber", "pymssql": "pymssql",
            "azure.storage.fileshare": "azure-storage-file-share", "pyspark": "pyspark"}
missing_packages = [pip_name for module, pip_name in packages.items() if not _has_module(module)]

required_env = ["AZURE_STORAGE_CONNECTION_STRING", "AZURE_FILE_SHARE_NAME",
                "MSSQL_HOST", "MSSQL_PORT", "MSSQL_DB", "MSSQL_USER", "MSSQL_PASSWORD"]
missing_env = [name for name in required_env if not os.environ.get(name)]

for name in required_env:
    print(f"  {name:<32} {'set' if os.environ.get(name) else 'MISSING'}")

if missing_packages:
    print(f"\nMissing packages: {missing_packages}")
    print("Fix: run `docker compose up -d --build jupyter` on the host (or, temporarily: %pip install " + " ".join(missing_packages) + ")")
if missing_env:
    print(f"\nMissing settings: {missing_env}")
    print("Fix: set them in the project's .env, then run `docker compose up -d jupyter` and restart this kernel.")
if not missing_packages and not missing_env:
    print("\nSetup OK.")

## 2. Imports

In [ ]:
import io
import re
import uuid
from concurrent.futures import ThreadPoolExecutor
from datetime import date, datetime, timedelta, timezone

import pandas as pd
import pdfplumber
import pymssql
from azure.core.exceptions import ResourceNotFoundError
from azure.storage.fileshare import ShareClient
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (DateType, DoubleType, IntegerType, StringType, StructField, StructType,
                               TimestampType)

## 3. Parameters
- `mode = "backfill"` processes every report in the `backfill_start` to `backfill_end` month range. Months with no folder are skipped.
- `mode = "single"` processes one report date, `target_date`. `None` means yesterday in UTC.
- `reprocess_existing = True` reloads reports that are already in SQL Server. The `MERGE` updates their rows instead of skipping them.

In [ ]:
mode = "single"            # "backfill" or "single"

# single mode: "YYYYMMDD", or None for yesterday (UTC)
target_date = "20261001"

# backfill mode: inclusive "YYYY-MM" range
# backfill_start = "2025-01"
# backfill_end   = "2026-09"

reprocess_existing = False   # True = reload reports already in datamart (MERGE updates them)

azure_root = os.environ.get("AZURE_LANDING_ROOT", "Grid_Report")   # folder inside the file share

# SQL Server schemas: datamart = final business tables only | staging = transient load tables | etl = pipeline bookkeeping
datamart_schema = "datamart"
staging_schema  = "staging"
etl_schema      = "etl"

summary_table     = f"{datamart_schema}.grid_operations_daily_summary"
detail_table      = f"{datamart_schema}.grid_operations_detail"
stg_summary_table = f"{staging_schema}.grid_operations_daily_summary"
stg_detail_table  = f"{staging_schema}.grid_operations_detail"
load_log_table    = f"{etl_schema}.file_load_log"

EXPECTED_ROWS_PER_REPORT = 24 * 4   # 24 hours x 4 substations
DOWNLOAD_WORKERS = 8
PIPELINE_NAME = "grid_report_backfill_to_sqlserver"
run_id = str(uuid.uuid4())

if mode not in ("backfill", "single"):
    raise ValueError(f"Unknown mode: {mode!r}")
if mode == "single" and target_date is None:
    target_date = (datetime.now(timezone.utc).date() - timedelta(days=1)).strftime("%Y%m%d")

print(f"Run id:  {run_id}")
print(f"Mode:    {mode}  " + (f"(target date {target_date})" if mode == "single" else f"({backfill_start} to {backfill_end})"))
print(f"Source:  Azure File Share '{os.environ.get('AZURE_FILE_SHARE_NAME')}'/{azure_root}")
print(f"Target:  SQL Server {os.environ.get('MSSQL_HOST')}:{os.environ.get('MSSQL_PORT')} / {os.environ.get('MSSQL_DB')}")
print(f"  final:   {summary_table}, {detail_table}")
print(f"  staging: {stg_summary_table}, {stg_detail_table}")
print(f"  log:     {load_log_table}")
print(f"Reprocess existing: {reprocess_existing}")

## 4. Connections (read-only checks)
Starts the Spark session and checks that the Azure File Share and SQL Server are reachable.

In [ ]:
spark = SparkSession.builder.appName(PIPELINE_NAME).getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")

# --- Azure File Share (landing layer)
share = ShareClient.from_connection_string(os.environ["AZURE_STORAGE_CONNECTION_STRING"],
                                           os.environ["AZURE_FILE_SHARE_NAME"])

# --- SQL Server: pymssql for DDL / MERGE / checks, JDBC for Spark's bulk write to staging
MSSQL = {
    "server":   os.environ["MSSQL_HOST"],
    "port":     os.environ.get("MSSQL_PORT", "1433"),
    "user":     os.environ["MSSQL_USER"],
    "password": os.environ["MSSQL_PASSWORD"],
    "database": os.environ["MSSQL_DB"],
}

def sql_connection(autocommit=True):
    return pymssql.connect(**MSSQL, autocommit=autocommit, login_timeout=15, timeout=300,
                           appname=PIPELINE_NAME)

def sql_df(query, params=None):
    """Runs a query and returns the result as a pandas DataFrame."""
    with sql_connection() as conn, conn.cursor() as cur:
        cur.execute(query, params)
        columns = [c[0] for c in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=columns)

JDBC_URL = (f"jdbc:sqlserver://{MSSQL['server']}:{MSSQL['port']};databaseName={MSSQL['database']};"
            "encrypt=true;trustServerCertificate=true")
JDBC_OPTIONS = {"url": JDBC_URL, "user": MSSQL["user"], "password": MSSQL["password"],
                "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"}

# --- checks
ui_port = spark.sparkContext.uiWebUrl.rsplit(":", 1)[-1]
print(f"Spark {spark.version} | master {spark.sparkContext.master} | app {spark.sparkContext.applicationId}")
if ui_port == "4040":
    print("Spark UI: http://localhost:4040")
else:
    print(f"Spark UI is on port {ui_port}, which isn't published to Windows (only 4040 is). Another Spark session "
          "is still running in this container; stop the other kernel to get port 4040 back.")
share_props = share.get_share_properties()
print(f"Azure File Share '{share_props.name}' reachable (quota {share_props.quota} GB)")
display(sql_df("SELECT @@SERVERNAME AS server_name, DB_NAME() AS database_name, SUSER_SNAME() AS login_name, "
               "CAST(SERVERPROPERTY('ProductVersion') AS nvarchar(32)) AS version"))

## 5. Target schemas and tables (idempotent DDL)
Creates three schemas and their tables **only if they don't exist yet**, so it's safe to re-run. Nothing is dropped or changed.

| Schema | Table | Grain | Key |
|---|---|---|---|
| **`datamart`**: final business tables only | `grid_operations_daily_summary` | 1 row per report (day) | `source_file` |
| | `grid_operations_detail` | 1 row per hour × substation (96 per report) | `source_file, reading_time, substation` |
| **`staging`**: transient load tables | `grid_operations_daily_summary`, `grid_operations_detail` | the same columns, no key | truncated and refilled each run |
| **`etl`**: pipeline bookkeeping | `file_load_log` | 1 row per processed file per run | `log_id` |

The column lists below are the single source of truth for the DDL, the Spark column order and the `MERGE`.

In [ ]:
SUMMARY_COLUMNS = [
    ("source_file",            "NVARCHAR(260) NOT NULL"),
    ("report_date",            "DATE NOT NULL"),
    ("grid_region",            "NVARCHAR(100) NULL"),
    ("operator",               "NVARCHAR(200) NULL"),
    ("balancing_authority",    "NVARCHAR(200) NULL"),
    ("transmission_operator",  "NVARCHAR(200) NULL"),
    ("market_zone",            "NVARCHAR(50) NULL"),
    ("generated_at",           "DATETIME2(0) NULL"),
    ("peak_load_mw",           "DECIMAL(10,1) NULL"),
    ("avg_load_mw",            "DECIMAL(10,1) NULL"),
    ("avg_frequency_hz",       "DECIMAL(7,3) NULL"),
    ("avg_reserve_margin_pct", "DECIMAL(9,1) NULL"),
    ("renewable_share_pct",    "DECIMAL(5,1) NULL"),
    ("avg_reliability_index",  "DECIMAL(5,1) NULL"),
    ("detail_row_count",       "INT NULL"),
]
SUMMARY_KEY = ["source_file"]

DETAIL_COLUMNS = [
    ("source_file",            "NVARCHAR(260) NOT NULL"),
    ("reading_time",           "CHAR(5) NOT NULL"),
    ("substation",             "NVARCHAR(150) NOT NULL"),
    ("report_date",            "DATE NOT NULL"),
    ("reading_datetime",       "DATETIME2(0) NOT NULL"),
    ("grid_region",            "NVARCHAR(100) NULL"),
    ("operator",               "NVARCHAR(200) NULL"),
    ("balancing_authority",    "NVARCHAR(200) NULL"),
    ("transmission_operator",  "NVARCHAR(200) NULL"),
    ("market_zone",            "NVARCHAR(50) NULL"),
    ("generated_at",           "DATETIME2(0) NULL"),
    ("gen_mw",                 "DECIMAL(10,1) NULL"),
    ("load_mw",                "DECIMAL(10,1) NULL"),
    ("renew_mw",               "DECIMAL(10,1) NULL"),
    ("wind_mw",                "DECIMAL(10,1) NULL"),
    ("solar_mw",               "DECIMAL(10,1) NULL"),
    ("batt_mw",                "DECIMAL(10,1) NULL"),
    ("freq_hz",                "DECIMAL(7,3) NULL"),
    ("volt_kv",                "DECIMAL(8,1) NULL"),
    ("congestion_pct",         "DECIMAL(5,1) NULL"),
    ("avail_cap_mw",           "DECIMAL(10,1) NULL"),
    ("reserve_pct",            "DECIMAL(9,1) NULL"),
    ("carbon_g_per_kwh",       "DECIMAL(8,1) NULL"),
    ("lmp_usd_per_mwh",        "DECIMAL(10,2) NULL"),
    ("weather",                "NVARCHAR(30) NULL"),
    ("status",                 "NVARCHAR(20) NULL"),
    ("reliability_index",      "DECIMAL(5,1) NULL"),
    ("renewable_pct_of_gen",   "DECIMAL(7,2) NULL"),
    ("is_operational",         "BIT NULL"),
]
DETAIL_KEY = ["source_file", "reading_time", "substation"]


def create_schema_sql(schema):
    return f"IF SCHEMA_ID(N'{schema}') IS NULL EXEC(N'CREATE SCHEMA [{schema}]');"


def create_table_sql(table, columns, key=None, audit_columns=False):
    base = table.split(".")[-1]
    lines = [f"[{name}] {sql_type}" for name, sql_type in columns]
    if audit_columns:
        lines += [f"[loaded_at] DATETIME2(0) NOT NULL CONSTRAINT [DF_{base}_loaded_at] DEFAULT SYSUTCDATETIME()",
                  "[updated_at] DATETIME2(0) NULL"]
    if key:
        lines.append(f"CONSTRAINT [PK_{base}] PRIMARY KEY CLUSTERED ({', '.join(f'[{k}]' for k in key)})")
    body = ",\n    ".join(lines)
    return f"IF OBJECT_ID(N'{table}', N'U') IS NULL\nCREATE TABLE {table} (\n    {body}\n);"


def create_index_sql(table, name, columns):
    return (f"IF NOT EXISTS (SELECT 1 FROM sys.indexes WHERE name = N'{name}' AND object_id = OBJECT_ID(N'{table}'))\n"
            f"CREATE INDEX [{name}] ON {table} ({', '.join(f'[{c}]' for c in columns)});")


DDL = [
    create_schema_sql(datamart_schema),
    create_schema_sql(staging_schema),
    create_schema_sql(etl_schema),
    # datamart: final business tables (keys + audit columns)
    create_table_sql(summary_table, SUMMARY_COLUMNS, key=SUMMARY_KEY, audit_columns=True),
    create_table_sql(detail_table, DETAIL_COLUMNS, key=DETAIL_KEY, audit_columns=True),
    create_index_sql(summary_table, "IX_grid_operations_daily_summary_report_date", ["report_date"]),
    create_index_sql(detail_table, "IX_grid_operations_detail_report_date", ["report_date", "substation"]),
    # staging: same columns, no keys (truncated and refilled each run)
    create_table_sql(stg_summary_table, SUMMARY_COLUMNS),
    create_table_sql(stg_detail_table, DETAIL_COLUMNS),
    # etl: per-file load log
    f"""IF OBJECT_ID(N'{load_log_table}', N'U') IS NULL
CREATE TABLE {load_log_table} (
    [log_id]       BIGINT IDENTITY(1,1) NOT NULL CONSTRAINT [PK_file_load_log] PRIMARY KEY,
    [run_id]       UNIQUEIDENTIFIER NOT NULL,
    [pipeline]     NVARCHAR(100) NOT NULL,
    [source_file]  NVARCHAR(260) NOT NULL,
    [report_date]  DATE NULL,
    [status]       NVARCHAR(20) NOT NULL,      -- loaded | warning | failed
    [detail_rows]  INT NULL,
    [message]      NVARCHAR(4000) NULL,
    [logged_at]    DATETIME2(0) NOT NULL CONSTRAINT [DF_file_load_log_logged_at] DEFAULT SYSUTCDATETIME()
);""",
]

with sql_connection(autocommit=True) as conn, conn.cursor() as cur:
    for statement in DDL:
        cur.execute(statement)

display(sql_df("""
    SELECT s.name AS [schema], t.name AS [table], SUM(p.rows) AS [rows]
    FROM sys.tables t
    JOIN sys.schemas s    ON s.schema_id = t.schema_id
    JOIN sys.partitions p ON p.object_id = t.object_id AND p.index_id IN (0, 1)
    WHERE s.name IN (%s, %s, %s)
    GROUP BY s.name, t.name
    ORDER BY s.name, t.name
""", (datamart_schema, staging_schema, etl_schema)))

### 5b. One-time cleanup of the old layout (safe to re-run)
The first version of this notebook created its staging tables and load log inside `datamart`. This cell removes those leftovers so `datamart` holds only the business tables:
- `datamart.stg_grid_operations_daily_summary` and `datamart.stg_grid_operations_detail` are transient copies of staging data, so they're dropped.
- `datamart.etl_file_load_log` is dropped **only if it's empty**. Otherwise it's kept and reported.

When none of these exist, the cell does nothing.

In [ ]:
legacy_staging = [f"{datamart_schema}.stg_grid_operations_daily_summary", f"{datamart_schema}.stg_grid_operations_detail"]
legacy_log = f"{datamart_schema}.etl_file_load_log"

with sql_connection(autocommit=True) as conn, conn.cursor() as cur:
    for table in legacy_staging:
        cur.execute(f"SELECT OBJECT_ID(N'{table}', N'U')")
        if cur.fetchone()[0] is None:
            print(f"  not present: {table}")
            continue
        cur.execute(f"DROP TABLE {table}")
        print(f"  dropped:     {table}")

    cur.execute(f"SELECT OBJECT_ID(N'{legacy_log}', N'U')")
    if cur.fetchone()[0] is None:
        print(f"  not present: {legacy_log}")
    else:
        cur.execute(f"SELECT COUNT(*) FROM {legacy_log}")
        rows = cur.fetchone()[0]
        if rows == 0:
            cur.execute(f"DROP TABLE {legacy_log}")
            print(f"  dropped:     {legacy_log} (was empty)")
        else:
            print(f"  KEPT:        {legacy_log} has {rows} rows; copy them to {load_log_table} if needed, then drop it manually")

## 6. Spark schemas
These are the columns the parser produces. The derived columns are added in step 12.

In [ ]:
METADATA_FIELDS = ["report_date", "grid_region", "generated_at", "operator",
                   "balancing_authority", "transmission_operator", "market_zone"]
SUMMARY_METRICS = ["peak_load_mw", "avg_load_mw", "avg_frequency_hz",
                   "avg_reserve_margin_pct", "renewable_share_pct", "avg_reliability_index"]

detail_schema = StructType([
    StructField("source_file",           StringType(),    False),
    StructField("report_date",           DateType(),      True),
    StructField("grid_region",           StringType(),    True),
    StructField("generated_at",          TimestampType(), True),
    StructField("operator",              StringType(),    True),
    StructField("balancing_authority",   StringType(),    True),
    StructField("transmission_operator", StringType(),    True),
    StructField("market_zone",           StringType(),    True),
    StructField("reading_time",          StringType(),    False),
    StructField("substation",            StringType(),    False),
    StructField("gen_mw",                DoubleType(),    True),
    StructField("load_mw",               DoubleType(),    True),
    StructField("renew_mw",              DoubleType(),    True),
    StructField("wind_mw",               DoubleType(),    True),
    StructField("solar_mw",              DoubleType(),    True),
    StructField("batt_mw",               DoubleType(),    True),
    StructField("freq_hz",               DoubleType(),    True),
    StructField("volt_kv",               DoubleType(),    True),
    StructField("congestion_pct",        DoubleType(),    True),
    StructField("avail_cap_mw",          DoubleType(),    True),
    StructField("reserve_pct",           DoubleType(),    True),
    StructField("carbon_g_per_kwh",      DoubleType(),    True),
    StructField("lmp_usd_per_mwh",       DoubleType(),    True),
    StructField("weather",               StringType(),    True),
    StructField("status",                StringType(),    True),
    StructField("reliability_index",     DoubleType(),    True),
])

summary_schema = StructType([
    StructField("source_file",            StringType(),    False),
    StructField("report_date",            DateType(),      True),
    StructField("grid_region",            StringType(),    True),
    StructField("generated_at",           TimestampType(), True),
    StructField("operator",               StringType(),    True),
    StructField("balancing_authority",    StringType(),    True),
    StructField("transmission_operator",  StringType(),    True),
    StructField("market_zone",            StringType(),    True),
    StructField("peak_load_mw",           DoubleType(),    True),
    StructField("avg_load_mw",            DoubleType(),    True),
    StructField("avg_frequency_hz",       DoubleType(),    True),
    StructField("avg_reserve_margin_pct", DoubleType(),    True),
    StructField("renewable_share_pct",    DoubleType(),    True),
    StructField("avg_reliability_index",  DoubleType(),    True),
    StructField("detail_row_count",       IntegerType(),   True),
])

## 7. Parsers: header/summary and detail rows
`pdfplumber` turns each report into lines like these:


In [ ]:
NUM = r"(-?\d+(?:\.\d+)?)"
GENERATED_PATTERN = r"Generated\s+(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})"


def _find(pattern, text, cast=str):
    match = re.search(pattern, text, re.MULTILINE)
    if not match:
        return None
    value = match.group(1).strip()
    return cast(value) if value else None


def parse_report_header(full_text: str, flow_text: str = "") -> dict:
    """full_text: page-position order. flow_text: written order; used for the footer's
    "Generated ..." stamp, which overlaps the disclaimer text in position order."""
    report_date = _find(r"Report Date:\s*(\d{4}-\d{2}-\d{2})", full_text)
    generated   = _find(GENERATED_PATTERN, flow_text) or _find(GENERATED_PATTERN, full_text)
    authorities = re.search(r"^Balancing Authority\s+(.+?)\s+Transmission Operator\s+(.+?)\s*$",
                            full_text, re.MULTILINE)
    return {
        "report_date":            datetime.strptime(report_date, "%Y-%m-%d").date() if report_date else None,
        "generated_at":           datetime.strptime(generated, "%Y-%m-%d %H:%M:%S") if generated else None,
        "grid_region":            _find(r"Grid Region:\s*(.+?)\s*$", full_text),
        "operator":               _find(r"^(.+?)\s+Grid Region:", full_text),
        "balancing_authority":    authorities.group(1).strip() if authorities else None,
        "transmission_operator":  authorities.group(2).strip() if authorities else None,
        "market_zone":            _find(r"^Market Zone\s+(.+?)\s+Reporting Period", full_text),
        "peak_load_mw":           _find(rf"Peak Load \(MW\)\s+{NUM}", full_text, float),
        "avg_load_mw":            _find(rf"Avg Load \(MW\)\s+{NUM}", full_text, float),
        "avg_frequency_hz":       _find(rf"Avg Frequency \(Hz\)\s+{NUM}", full_text, float),
        "avg_reserve_margin_pct": _find(rf"Avg Reserve Margin \(%\)\s+{NUM}", full_text, float),
        "renewable_share_pct":    _find(rf"Renewable Share \(%\)\s+{NUM}", full_text, float),
        "avg_reliability_index":  _find(rf"Avg Reliability Index\s+{NUM}", full_text, float),
    }


# time, substation, 13 numbers, "weather status" text, reliability
DETAIL_ROW_PATTERN = re.compile(
    r"^(\d{2}:\d{2})\s+"
    r"(.+?)\s+"
    r"(-?[\d.]+)\s+(-?[\d.]+)\s+(-?[\d.]+)\s+(-?[\d.]+)\s+"
    r"(-?[\d.]+)\s+(-?[\d.]+)\s+(-?[\d.]+)\s+(-?[\d.]+)\s+"
    r"(-?[\d.]+)\s+(-?[\d.]+)\s+(-?[\d.]+)\s+(-?[\d.]+)\s+"
    r"(-?[\d.]+)\s+"
    r"([A-Za-z ]+?)\s+"
    r"([\d.]+)$"
)
DETAIL_NUMERIC_FIELDS = ["gen_mw", "load_mw", "renew_mw", "wind_mw", "solar_mw", "batt_mw", "freq_hz", "volt_kv",
                         "congestion_pct", "avail_cap_mw", "reserve_pct", "carbon_g_per_kwh", "lmp_usd_per_mwh"]
KNOWN_STATUSES = ("Operational", "Maintenance", "Alert")


def split_weather_status(text: str) -> tuple:
    """'Partly Cloudy' overflows its column, so weather and status can arrive glued together
    ('Partly CloudyMaintenance'). Split off the known status; fall back to the last word."""
    text = text.strip()
    for status in KNOWN_STATUSES:
        if text.endswith(status):
            return text[:-len(status)].strip(), status
    weather, _, status = text.rpartition(" ")
    return weather.strip(), status


def parse_detail_rows(full_text: str) -> list:
    rows = []
    for line in full_text.splitlines():
        match = DETAIL_ROW_PATTERN.match(line.strip())
        if not match:
            continue
        g = match.groups()
        weather, status = split_weather_status(g[15])
        row = {"reading_time": g[0], "substation": g[1].strip()}
        row.update({name: float(value) for name, value in zip(DETAIL_NUMERIC_FIELDS, g[2:15])})
        row.update({"weather": weather, "status": status, "reliability_index": float(g[16])})
        rows.append(row)
    return rows

## 8. Per-file extractor
Turns one PDF into its detail rows (96 expected) and one summary row. A file whose report date can't be found, or that has no detail rows, raises an error. That file is logged as `failed` and the run carries on.

In [ ]:
def extract_grid_report(file_bytes: bytes, file_name: str) -> dict:
    with pdfplumber.open(io.BytesIO(file_bytes)) as pdf:
        # Header/summary: text in page-position order (keeps "<operator> Grid Region: <region>" on one line).
        layout_text = "\n".join(page.extract_text() or "" for page in pdf.pages)
        # Hourly rows + footer timestamp: text in the order it was written to the PDF. Long substation names
        # overflow into the Gen (MW) column (and the footer stamp overlaps the disclaimer); position order
        # interleaves them ("Switching Stat3io6n6.2"), written order doesn't.
        flow_text = "\n".join(page.extract_text(use_text_flow=True) or "" for page in pdf.pages)

    header = parse_report_header(layout_text, flow_text)
    if header["report_date"] is None:
        raise ValueError("'Report Date' not found; not a grid report?")
    detail_rows = parse_detail_rows(flow_text)
    if not detail_rows:
        raise ValueError("no hourly detail rows matched")

    metadata = {field: header[field] for field in METADATA_FIELDS}
    for row in detail_rows:
        row.update(source_file=file_name, **metadata)

    summary_row = {"source_file": file_name, **metadata,
                   **{metric: header[metric] for metric in SUMMARY_METRICS},
                   "detail_row_count": len(detail_rows)}
    return {"detail": detail_rows, "summary": summary_row}

## 9. Build the list of folders and find the PDFs (read-only)
Lists `Grid_Report/{YYYY}/{MM}/` in the file share. Missing month folders are skipped, and in-progress `.partial` uploads are ignored.

In single mode, each file name is split as `{region}_{YYYYMMDD}_grid_report.pdf`, and every report whose date matches is kept.

In [ ]:
REPORT_NAME_RE = re.compile(r"^(?P<region>.+)_(?P<date>\d{8})_grid_report\.pdf$", re.IGNORECASE)


def month_range(start_ym: str, end_ym: str):
    """Yields (YYYY, MM) from start to end inclusive, e.g. '2025-11' -> '2026-02'."""
    current = datetime.strptime(start_ym, "%Y-%m")
    end = datetime.strptime(end_ym, "%Y-%m")
    while current <= end:
        yield current.strftime("%Y"), current.strftime("%m")
        current = current.replace(year=current.year + (current.month == 12), month=current.month % 12 + 1)


def list_month(year: str, month: str):
    """PDF paths in one month folder of the share, or None if the folder doesn't exist."""
    folder = f"{azure_root}/{year}/{month}"
    try:
        items = list(share.get_directory_client(folder).list_directories_and_files())
    except ResourceNotFoundError:
        return None
    return sorted(f"{folder}/{item.name}" for item in items
                  if not item.is_directory and item.name.lower().endswith(".pdf"))


if mode == "single":
    months = [(target_date[0:4], target_date[4:6])]
else:
    months = list(month_range(backfill_start, backfill_end))

candidates, missing_folders = [], []
for year, month in months:
    paths = list_month(year, month)
    if paths is None:
        missing_folders.append(f"{year}/{month}")
        continue
    for path in paths:
        name = path.rsplit("/", 1)[-1]
        match = REPORT_NAME_RE.match(name)
        if mode == "single" and (not match or match["date"] != target_date):
            continue
        candidates.append({"path": path, "name": name})
    print(f"  {azure_root}/{year}/{month}: {len(paths)} PDF(s)")

if missing_folders:
    print(f"Skipped missing folder(s): {', '.join(missing_folders)}")
print(f"\nReports found in the landing layer: {len(candidates)}")

## 10. Filter out reports already loaded (incremental)
A report counts as loaded when its `source_file` is in `datamart.grid_operations_daily_summary`. With `reprocess_existing = True`, everything found is processed again.

In [ ]:
already_loaded = set()
if not reprocess_existing:
    already_loaded = set(sql_df(f"SELECT source_file FROM {summary_table}")["source_file"])

files_to_process = [f for f in candidates if f["name"] not in already_loaded]

print(f"Already in {summary_table}: {len(already_loaded)}")
print(f"Reports to process this run: {len(files_to_process)}")
if not files_to_process:
    print("\nNothing new to load. The remaining cells will skip their work.")

## 11. Download and parse
Downloads are parallel (`DOWNLOAD_WORKERS`) and parsing runs on the driver, as in the Fabric notebook. A file that fails is recorded with its error and the run continues. A report with a row count other than 96 is kept, but flagged as `warning`.

In [ ]:
def fetch(file_info):
    try:
        content = share.get_file_client(file_info["path"]).download_file().readall()
        return file_info, content, None
    except Exception as exc:  # noqa: BLE001 - recorded per file
        return file_info, None, exc


detail_records, summary_records, file_results = [], [], []

if files_to_process:
    with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
        for i, (file_info, content, error) in enumerate(pool.map(fetch, files_to_process), 1):
            name = file_info["name"]
            if error is not None:
                file_results.append({"source_file": name, "report_date": None, "status": "failed",
                                     "detail_rows": None, "message": f"download: {error}"[:4000]})
            else:
                try:
                    result = extract_grid_report(content, name)
                    rows = len(result["detail"])
                    detail_records.extend(result["detail"])
                    summary_records.append(result["summary"])
                    ok = rows == EXPECTED_ROWS_PER_REPORT
                    file_results.append({"source_file": name, "report_date": result["summary"]["report_date"],
                                         "status": "loaded" if ok else "warning", "detail_rows": rows,
                                         "message": None if ok else f"expected {EXPECTED_ROWS_PER_REPORT} detail rows, parsed {rows}"})
                except Exception as exc:  # noqa: BLE001 - recorded per file
                    file_results.append({"source_file": name, "report_date": None, "status": "failed",
                                         "detail_rows": None, "message": f"parse: {exc}"[:4000]})
            if i % 25 == 0 or i == len(files_to_process):
                print(f"  processed {i}/{len(files_to_process)} files...")

failed = [r for r in file_results if r["status"] == "failed"]
warnings = [r for r in file_results if r["status"] == "warning"]
print(f"\nParsed reports: {len(summary_records)} | detail rows: {len(detail_records)} | "
      f"warnings: {len(warnings)} | failed: {len(failed)}")
for r in failed + warnings:
    print(f"  {r['status'].upper():<8} {r['source_file']}: {r['message']}")

## 12. Spark DataFrames and transformations
The derived columns are the same as in the Fabric notebook:
- `reading_datetime` = report date + reading time.
- `renewable_pct_of_gen` = renewables ÷ generation × 100. It's null when generation is 0.
- `is_operational` = whether status is `Operational`.

Rows are also de-duplicated on the table keys, and the columns are put in the SQL tables' order.

In [ ]:
if detail_records:
    detail_df = spark.createDataFrame(detail_records, schema=detail_schema)
    summary_df = spark.createDataFrame(summary_records, schema=summary_schema)

    detail_df = (
        detail_df
        .withColumn("reading_datetime", F.to_timestamp(
            F.concat_ws(" ", F.col("report_date").cast("string"), F.col("reading_time")), "yyyy-MM-dd HH:mm"))
        .withColumn("renewable_pct_of_gen",
                    F.when(F.col("gen_mw") > 0, F.round(F.col("renew_mw") / F.col("gen_mw") * 100, 2)))
        .withColumn("is_operational", F.col("status") == F.lit("Operational"))
        .dropDuplicates(DETAIL_KEY)
        .select(*[name for name, _ in DETAIL_COLUMNS])
    )
    summary_df = summary_df.dropDuplicates(SUMMARY_KEY).select(*[name for name, _ in SUMMARY_COLUMNS])

    print(f"detail_df: {detail_df.count()} rows | summary_df: {summary_df.count()} rows")
    detail_df.show(5, truncate=False)
else:
    print("No new records; nothing to transform.")

## 13. Write to the `staging` tables (JDBC), the first write to SQL Server
`staging.grid_operations_daily_summary` and `staging.grid_operations_detail` are emptied with `TRUNCATE` and then appended to. Spark's executors write in parallel through the SQL Server JDBC driver that's built into the image. The `datamart` tables aren't touched until step 14.

In [ ]:
def write_staging(df, table, partitions):
    with sql_connection(autocommit=True) as conn, conn.cursor() as cur:
        cur.execute(f"TRUNCATE TABLE {table}")
    (df.repartition(partitions)
       .write.format("jdbc")
       .options(**JDBC_OPTIONS)
       .option("dbtable", table)
       .option("batchsize", 5000)
       .mode("append")
       .save())


staged = False
if detail_records:
    write_staging(summary_df, stg_summary_table, partitions=1)
    write_staging(detail_df, stg_detail_table, partitions=2)
    staged = True
    display(sql_df(f"SELECT '{stg_summary_table}' AS staging_table, COUNT(*) AS rows FROM {stg_summary_table} "
                   f"UNION ALL SELECT '{stg_detail_table}', COUNT(*) FROM {stg_detail_table}"))
else:
    print("No new records; staging not touched.")

## 14. MERGE `staging` → `datamart` (one transaction): the final stage
Upserts both business tables in a single transaction: either both succeed or neither changes.
- **New keys are inserted.**
- **Existing keys are updated** and get `updated_at`. This only happens with `reprocess_existing = True`, because otherwise already-loaded reports never reach staging.

Staging keeps its data after the merge so you can inspect it. It's emptied again at the start of the next run.

In [ ]:
def merge_sql(target, staging, columns, key):
    names = [name for name, _ in columns]
    on = " AND ".join(f"t.[{k}] = s.[{k}]" for k in key)
    updates = ",\n        ".join([f"t.[{n}] = s.[{n}]" for n in names if n not in key] + ["t.[updated_at] = SYSUTCDATETIME()"])
    insert_cols = ", ".join(f"[{n}]" for n in names)
    insert_vals = ", ".join(f"s.[{n}]" for n in names)
    return f"""
SET NOCOUNT ON;
DECLARE @actions TABLE ([action] NVARCHAR(10));
MERGE {target} WITH (HOLDLOCK) AS t
USING {staging} AS s
   ON {on}
WHEN MATCHED THEN UPDATE SET
        {updates}
WHEN NOT MATCHED BY TARGET THEN
    INSERT ({insert_cols})
    VALUES ({insert_vals})
OUTPUT $action INTO @actions;
SELECT [action], COUNT(*) AS [rows] FROM @actions GROUP BY [action];
"""


merge_completed = False
merge_results = {}
if staged:
    conn = sql_connection(autocommit=False)
    try:
        with conn.cursor() as cur:
            for target, staging, columns, key in [(summary_table, stg_summary_table, SUMMARY_COLUMNS, SUMMARY_KEY),
                                                  (detail_table, stg_detail_table, DETAIL_COLUMNS, DETAIL_KEY)]:
                cur.execute(merge_sql(target, staging, columns, key))
                merge_results[target] = {action: rows for action, rows in cur.fetchall()}
        conn.commit()
        merge_completed = True
    except Exception:
        conn.rollback()
        print("MERGE failed; transaction rolled back, datamart unchanged.")
        raise
    finally:
        conn.close()

    for target, actions in merge_results.items():
        print(f"{target}: inserted {actions.get('INSERT', 0)}, updated {actions.get('UPDATE', 0)}")
else:
    print("Nothing staged; MERGE skipped.")

## 15. Write the load log (`etl.file_load_log`)
One row per processed file for this `run_id`:
- **`failed`** files are always logged.
- **`loaded`/`warning`** files are logged only once the MERGE has committed.

In [ ]:
log_rows = [r for r in file_results if r["status"] == "failed" or merge_completed]

if log_rows:
    with sql_connection(autocommit=True) as conn, conn.cursor() as cur:
        cur.executemany(
            f"INSERT INTO {load_log_table} (run_id, pipeline, source_file, report_date, status, detail_rows, message) "
            "VALUES (%s, %s, %s, %s, %s, %s, %s)",
            [(run_id, PIPELINE_NAME, r["source_file"], r["report_date"], r["status"], r["detail_rows"], r["message"])
             for r in log_rows])
    display(sql_df(f"SELECT status, COUNT(*) AS files, SUM(detail_rows) AS detail_rows "
                   f"FROM {load_log_table} WHERE run_id = %s GROUP BY status", (run_id,)))
else:
    print("Nothing to log for this run.")

## 16. Data quality checks (read-only)
These run against the whole `datamart` business tables, not only this run. Every check should return **0 rows** or **0**.

In [ ]:
checks = {
    "Reports whose detail row count isn't 96": f"""
        SELECT source_file, COUNT(*) AS detail_rows FROM {detail_table}
        GROUP BY source_file HAVING COUNT(*) <> {EXPECTED_ROWS_PER_REPORT}""",
    "Summary rows without detail rows": f"""
        SELECT s.source_file FROM {summary_table} s
        WHERE NOT EXISTS (SELECT 1 FROM {detail_table} d WHERE d.source_file = s.source_file)""",
    "Detail rows without a summary row": f"""
        SELECT DISTINCT d.source_file FROM {detail_table} d
        WHERE NOT EXISTS (SELECT 1 FROM {summary_table} s WHERE s.source_file = d.source_file)""",
    "summary.detail_row_count differs from the actual detail rows": f"""
        SELECT s.source_file, s.detail_row_count, COUNT(d.source_file) AS actual
        FROM {summary_table} s LEFT JOIN {detail_table} d ON d.source_file = s.source_file
        GROUP BY s.source_file, s.detail_row_count HAVING COUNT(d.source_file) <> s.detail_row_count""",
}
for title, query in checks.items():
    result = sql_df(query)
    print(f"[{'OK' if result.empty else 'CHECK'}] {title}: {len(result)} row(s)")
    if not result.empty:
        display(result.head(20))

print("\nNull counts in header fields (expect all 0):")
display(sql_df(f"""
    SELECT COUNT(*) AS reports,
           SUM(CASE WHEN operator              IS NULL THEN 1 ELSE 0 END) AS null_operator,
           SUM(CASE WHEN balancing_authority   IS NULL THEN 1 ELSE 0 END) AS null_balancing_authority,
           SUM(CASE WHEN transmission_operator IS NULL THEN 1 ELSE 0 END) AS null_transmission_operator,
           SUM(CASE WHEN market_zone           IS NULL THEN 1 ELSE 0 END) AS null_market_zone,
           SUM(CASE WHEN generated_at          IS NULL THEN 1 ELSE 0 END) AS null_generated_at,
           SUM(CASE WHEN peak_load_mw          IS NULL THEN 1 ELSE 0 END) AS null_peak_load
    FROM {summary_table}"""))

## 17. Look at the data (read-only)

In [ ]:
print("Reports per month:")
display(sql_df(f"""
    SELECT FORMAT(report_date, 'yyyy-MM') AS [month], COUNT(*) AS reports,
           MIN(report_date) AS first_report, MAX(report_date) AS last_report,
           CAST(AVG(renewable_share_pct) AS DECIMAL(5,1)) AS avg_renewable_share_pct,
           MAX(peak_load_mw) AS max_peak_load_mw
    FROM {summary_table}
    GROUP BY FORMAT(report_date, 'yyyy-MM')
    ORDER BY [month]"""))

print("Latest report, hourly totals across substations:")
display(sql_df(f"""
    SELECT TOP (24) d.reading_time,
           SUM(d.load_mw) AS load_mw, SUM(d.gen_mw) AS gen_mw, SUM(d.solar_mw) AS solar_mw,
           SUM(d.wind_mw) AS wind_mw, AVG(d.lmp_usd_per_mwh) AS avg_lmp_usd_per_mwh
    FROM {detail_table} d
    WHERE d.report_date = (SELECT MAX(report_date) FROM {summary_table})
    GROUP BY d.reading_time
    ORDER BY d.reading_time"""))

## 18. Finish
Releases the notebook's Spark executors, giving their 4 cores back to the cluster. Re-run step 4 to start again.

In [ ]:
spark.stop()
print(f"Run {run_id} finished.")